In [1]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Authenticate
login(hf_token)

In [2]:
import gc
import torch
import math
import numpy as np
import json
from glob import glob
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

# --- FUNÇÃO AUXILIAR PARA LIMPAR VRAM ---
def limpar_memoria():
    """Força a limpeza da memória RAM e VRAM (GPU)."""
    gc.collect()                # Limpa objetos Python não utilizados
    torch.cuda.empty_cache()    # Limpa o cache de alocação do PyTorch
    print("🧹 Memória limpa!")

# --- Configuração do Dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ambiente de Execução: {device.upper()}")

🚀 Ambiente de Execução: CUDA


In [3]:
def load_model_and_tokenizer(model_name:str, model_type:AutoModelForCausalLM | AutoModelForMaskedLM):
    if model_type is not AutoModelForCausalLM and model_type is not AutoModelForMaskedLM:
        raise Exception("Model type must be `AutoModelForCausalLM` or `AutoModelForMaskedLM` ")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = model_type.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )
    model.eval()

    return tokenizer, model


def eval_causal_sliding(model, tokenizer, dataset, stride=512):
    results = {}

    max_len = model.config.max_position_embeddings

    # The perplexity is calculated for each text
    for label, texts in dataset.items():
        text_results = []

        for i, text in enumerate(texts):

            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            num_tokens_text = inputs.input_ids.size(1)

            nlls = []

            total_nll = 0.0
            total_tokens = 0
            prev_end_window = 0

            # Sliding window
            for begin_window in range(0, num_tokens_text, stride):
                end_window = min(begin_window + max_len, num_tokens_text)

                # Since the windows overlap, this indicates how many new tokens
                # will be actually used to calculate the perplexity
                target_len = end_window - prev_end_window

                # Select only the tokens from the window
                input_ids = inputs.input_ids[:, begin_window:end_window]
                target_ids = input_ids.clone()

                # Loss will ignore labels with value -100
                target_ids[:, :-target_len] = -100

                with torch.no_grad():
                    outputs = model(input_ids, labels=target_ids)
                    neg_log_likelihood = outputs.loss

                total_nll += neg_log_likelihood.item() * target_len
                total_tokens += target_len
                prev_end_window = end_window

                # nlls.append(neg_log_likelihood)
                if end_window == num_tokens_text:
                    break

            ppl = math.exp(total_nll / total_tokens)

            text_results.append({
                "text_id": i,
                "num_tokens": target_len,
                "ppl": ppl
            })

        avg_ppl = sum(r["ppl"] for r in text_results) / len(text_results)

        results[label] = {
            "avg_ppl": avg_ppl,
            "texts": text_results
        }

    return results


In [4]:
CORPORA_FOLDER = "./corpora/"
OUTPUT_FOLDER = "./results/"

models = {
    "Gemma": "google/gemma-2-2b-it",
    "Qwen" : "Qwen/Qwen3-Embedding-0.6B"
}

In [5]:
def execute_ppl_experiment_casualLM(model_name, model_link):

    print(f"Executing {model_name}")

    tokenizer, model = load_model_and_tokenizer(model_link, AutoModelForCausalLM)

    corpora_files = glob(f"{CORPORA_FOLDER}*.json")
    corpora_files.sort()

    for file_path in corpora_files:
        corpora_name = Path(file_path).stem

        print(f"Processing corpora: {corpora_name}")

        with open(file_path, "r", encoding="utf-8") as f:
            dataset = json.load(f)

        results = eval_causal_sliding(model, tokenizer, dataset)

        print(f"\n{'Language':>10} | {'PPL':>10}")
        print(f"-"*25)
        for language in results.keys():
            print(f"{language:>10} | {results[language]["avg_ppl"] :> 10.3f}")


        results_file = f"{OUTPUT_FOLDER}{model_name}-{corpora_name}.json"
        print(f"\nSaving results to: {results_file}")

        with open(results_file, 'w') as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

        print("\n", "="*50, "\n")

    del model
    del tokenizer
    limpar_memoria()

    return results


In [6]:
gemma_results = execute_ppl_experiment_casualLM("Gemma", models["Gemma"])

Executing Gemma


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Processing corpora: CAPACITAR

  Language |        PPL
--------------------
  ESPANHOL |     70.410
   FRANCÊS |     52.128
    GALEGO |    152.703
    INGLÊS |     75.735
  MANDARIM |     80.847
 PB NATIVO |     61.201
     RUSSO |     67.397
     ÁRABE |    115.090

Saving results to: ./results/Gemma-CAPACITAR.json


Processing corpora: COMPARTILHAR

  Language |        PPL
--------------------
  ESPANHOL |     71.485
   FRANCÊS |    109.163
    GALEGO |    144.945
    INGLÊS |     99.187
  MANDARIM |     36.158
 PB NATIVO |     68.053
     RUSSO |    114.063
     ÁRABE |    153.153

Saving results to: ./results/Gemma-COMPARTILHAR.json


Processing corpora: EXPLICAR

  Language |        PPL
--------------------
  ESPANHOL |     52.247
   FRANCÊS |     34.962
    GALEGO |    102.046
    INGLÊS |     54.476
  MANDARIM |     22.493
 PB NATIVO |     44.606
     RUSSO |     66.359
     ÁRABE |    122.678

Saving results to: ./results/Gemma-EXPLICAR.json


Processing corpora: EXPLORAR

  L